In [1]:
import numpy as np
import pandas as pd
from tqdm.autonotebook import tqdm
import torch, os, gc
from torch.optim import AdamW
import torch.nn as nn
from torch.nn.functional import binary_cross_entropy_with_logits as BCE

# to do cross-validation on the factor model dimension + choice of lmbda
from sklearn.model_selection import KFold

# set our device
device = "cuda" if torch.cuda.is_available() else "cpu"

/var/folders/nb/9r9j6r9d7598f8fwysr8yx6r0000gn/T/ipykernel_60355/1091806703.py:3: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [8]:
# load in our data (MMLU-Pro + GPQA-all): note that only baselines have access to GPQA values.
M1 = pd.read_csv(f"data/mmlu-pro-and-gpqa-all/M1_p=0.001.csv")
M1_mmlu_pro = torch.tensor(M1[[c for c in M1.columns if "mmlu_pro" in c]].to_numpy().astype(np.float32))

# move to the right device
M1_mmlu_pro = M1_mmlu_pro.to(device)

# size of the MMLU-Pro historical data?
N_M1_mmlu_pro, TOTAL_QUESTIONS_mmlu_pro = M1_mmlu_pro.shape

# load the LLM embeddings for our MMLU-Pro and GPQA (all) - both already normalized to unit vectors!
E_gpqa_all = torch.tensor(np.load("embeddings/gpqa_all_embeddings.npy"), device=device)
E_mmlu_pro = torch.tensor(np.load("embeddings/mmlu_pro_embeddings.npy"), device=device)

# get the cosine similarities
cossims = (E_gpqa_all @ E_mmlu_pro.T).float()

# how many iterations are we doing
MAX_EPOCHS = 2000

In [9]:
# hyperparameter settings
D, lmbda, K, = 8, 1.0, 8

# seed for reproducibility + init tensors & optimizer
torch.random.manual_seed(858)
U = nn.Parameter(torch.randn(size=(N_M1_mmlu_pro, D), requires_grad=True, device=device))
V = nn.Parameter(torch.randn(size=(TOTAL_QUESTIONS_mmlu_pro, D), requires_grad=True, device=device))
optimizer = AdamW(lr=1e-2, params=[{"params" : [U, V], "weight_decay" : lmbda}])

# initial fit on MMLU-Pro itself: forward pass, backwards pass, update parameters, reset gradient
for epoch in tqdm(range(MAX_EPOCHS), desc=f"MMLU-Pro fit w/t D={D}, lmbda={lmbda}"):
    mmlu_pro_loss = BCE(U @ V.T, M1_mmlu_pro); mmlu_pro_loss.backward(); optimizer.step(); optimizer.zero_grad()
U.requires_grad, V.requires_grad = False, False

# lock out U_gpqa (freeze at MMLU latent space values)
U_gpqa = U.clone()

# only build GPQA question factors based on LLM-embeddings KNN.
topk = cossims.topk(k=K, dim=1)
V_gpqa = torch.einsum("ijk,ij->ik", V[topk.indices], topk.values) / topk.values.sum(dim=1, keepdim=True)

# turn off gradients, save our reuslts.
U_gpqa.requires_grad, V_gpqa.requires_grad = False, False
torch.save(U_gpqa, f"factor_models/final/gpqa-all-transfer-0/U.pt")
torch.save(V_gpqa, f"factor_models/final/gpqa-all-transfer-0/V.pt")

MMLU-Pro fit w/t D=8, lmbda=1.0:   0%|          | 0/2000 [00:00<?, ?it/s]

D=8, lmbda=1.0, K=8, eta=0.001:   0%|          | 0/2000 [00:00<?, ?it/s]